# Mini-Project — MCP + Agents AI Integration with Gemini

**Week 10 · Day 5 — Bootcamp GenAI & Machine Learning 2026**

## Theme: Research Assistant

An end-to-end agentic application that orchestrates multiple MCP servers using Gemini. The agent autonomously decides which tools to call — no hard-coded flows.

### Architecture

```
User Query
    └──► LangGraph ReAct Agent (Gemini)
              │
              ├──► MCP Server 1: Filesystem  (read/write research notes)
              ├──► MCP Server 2: Memory      (persist key facts)
              └──► MCP Server 3: Custom Ops  (citation formatter + text analysis)
                        ↓
                  Final answer
```

| Server | Tools | Transport |
|--------|-------|-----------|
| `@modelcontextprotocol/server-filesystem` | read_file, write_file, list_dir | stdio / npx |
| Custom `research_ops` (FastMCP) | `format_citation`, `count_words`, `extract_keywords`, `ping` | stdio / python |
| Stub servers | same tools, no npm needed | inline Python |

---
## Step 1 — Install Dependencies

In [ ]:
%pip install -qU \
  "langchain>=0.3" \
  "langgraph>=0.2" \
  "langchain-google-genai>=2.0" \
  "google-genai>=1.0" \
  "langchain-mcp-adapters==0.2.1" \
  "fastmcp>=2.0.0" \
  "nest_asyncio" \
  "mcp[cli]"

---
## Step 2 — Configuration

In [ ]:
import os
import asyncio
import nest_asyncio
nest_asyncio.apply()  # allow asyncio in Jupyter/Colab

# --- API Key ---
# In Colab: Secrets → add GOOGLE_API_KEY
try:
    from google.colab import userdata
    GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")
except Exception:
    GOOGLE_API_KEY = os.environ.get("GOOGLE_API_KEY", "")

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY or ""

USE_REAL_LLM = bool(GOOGLE_API_KEY)   # auto-detect
USE_NPM_SERVERS = False               # set True if npx is available

# Working directory for the research assistant
WORKDIR = "/content/research" if os.path.exists("/content") else "/tmp/research"
os.makedirs(WORKDIR, exist_ok=True)

print(f"LLM       : {'Gemini (real)' if USE_REAL_LLM else 'Stub (no key)'}")
print(f"Servers   : {'npm + custom' if USE_NPM_SERVERS else 'custom Python only'}")
print(f"WORKDIR   : {WORKDIR}")

---
## Step 3 — Node/NPM Check

In [ ]:
import subprocess

def check_node():
    try:
        node_v = subprocess.check_output(["node", "--version"], text=True).strip()
        npx_v  = subprocess.check_output(["npx", "--version"],  text=True).strip()
        print(f"node: {node_v}  |  npx: {npx_v}")
        return True
    except FileNotFoundError:
        print("node/npx not found — using Python-only MCP servers.")
        return False

npm_available = check_node()
USE_NPM_SERVERS = USE_NPM_SERVERS and npm_available

# Uncomment to install node on Colab if missing:
# !apt-get -qq update && apt-get -qq install -y nodejs npm
# npm_available = check_node()

---
## Step 4 — Workspace Setup

In [ ]:
from pathlib import Path

# Seed the workspace with sample research notes
notes = {
    "llm_overview.txt": (
        "Large Language Models (LLMs) are neural networks trained on vast text corpora.\n"
        "They demonstrate emergent capabilities: reasoning, code generation, instruction following.\n"
        "Key models: GPT-4 (OpenAI), Claude (Anthropic), Gemini (Google), LLaMA (Meta).\n"
        "Training uses self-supervised next-token prediction at massive scale."
    ),
    "rag_notes.txt": (
        "RAG (Retrieval-Augmented Generation) combines a retriever with a generator.\n"
        "Steps: embed query → retrieve top-k docs → generate answer conditioned on docs.\n"
        "Benefits: reduces hallucination, enables up-to-date knowledge, citable sources.\n"
        "Tools: FAISS, Chroma, Pinecone for vector stores; LangChain for orchestration."
    ),
    "mcp_notes.txt": (
        "MCP (Model Context Protocol) by Anthropic standardizes AI-to-tool connections.\n"
        "Architecture: Host → Client → Server. STDIO transport for local dev.\n"
        "Servers expose Tools (actions) and Resources (read-only context).\n"
        "FastMCP makes server creation trivial with decorators: @mcp.tool, @mcp.resource."
    ),
}

for fname, content in notes.items():
    (Path(WORKDIR) / fname).write_text(content)

print(f"Workspace seeded ✅ — {len(notes)} files in {WORKDIR}")
for f in Path(WORKDIR).iterdir():
    print(f"  {f.name}")

---
## Step 5 — Custom MCP Server (FastMCP)

A Python MCP server exposing research-specific tools: citation formatter, word counter, keyword extractor.

In [ ]:
import textwrap

server_path = Path(WORKDIR) / "research_ops_server.py"
server_path.write_text(textwrap.dedent("""
    from fastmcp import FastMCP
    from typing import Dict, List
    import re
    from collections import Counter

    mcp = FastMCP(name="research_ops")


    @mcp.tool
    def ping() -> str:
        \"\"\"Health check — returns 'pong'.\"\"\"
        return "pong"


    @mcp.tool
    def format_citation(title: str, author: str, year: int, venue: str, url: str = "") -> str:
        \"\"\"Format an APA-style academic citation from provided metadata.\"\"\"
        citation = f"{author} ({year}). {title}. {venue}."
        if url:
            citation += f" Retrieved from {url}"
        return citation


    @mcp.tool
    def count_words(text: str) -> Dict[str, int]:
        \"\"\"Count words, sentences, and paragraphs in a text.\"\"\"
        words = len(text.split())
        sentences = len([s for s in re.split(r'[.!?]+', text) if s.strip()])
        paragraphs = len([p for p in text.split('\\n\\n') if p.strip()])
        return {"words": words, "sentences": sentences, "paragraphs": paragraphs}


    @mcp.tool
    def extract_keywords(text: str, n: int = 5) -> List[str]:
        \"\"\"Extract the top-N most frequent meaningful words from a text.\"\"\"
        stopwords = {"the", "a", "an", "and", "or", "but", "in", "on", "at",
                     "to", "for", "of", "is", "are", "was", "with", "by", "it",
                     "this", "that", "from", "as", "be", "its", "they", "their"}
        words = re.findall(r'\\b[a-zA-Z]{4,}\\b', text.lower())
        freq = Counter(w for w in words if w not in stopwords)
        return [word for word, _ in freq.most_common(n)]


    @mcp.tool
    def summarize_lines(lines: List[str]) -> Dict[str, int]:
        \"\"\"Return line count statistics for a list of text lines.\"\"\"
        total = len(lines)
        nonempty = sum(1 for line in lines if line.strip())
        return {"total_lines": total, "nonempty_lines": nonempty}


    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print(f"Custom server written ✅ → {server_path}")

---
## Step 6 — Filesystem MCP Server

A Python-based filesystem server (no npm required) that exposes file read/write/list tools.

In [ ]:
fs_server_path = Path(WORKDIR) / "filesystem_server.py"
fs_server_path.write_text(textwrap.dedent(f"""
    from fastmcp import FastMCP
    from pathlib import Path
    from typing import List

    mcp = FastMCP(name="filesystem")
    WORKDIR = "{WORKDIR}"


    @mcp.tool
    def list_files() -> List[str]:
        \"\"\"List all files in the research workspace.\"\"\"
        return [f.name for f in Path(WORKDIR).iterdir() if f.is_file() and not f.name.endswith('.py')]


    @mcp.tool
    def read_file(filename: str) -> str:
        \"\"\"Read the content of a file from the research workspace.\"\"\"
        path = Path(WORKDIR) / filename
        if not path.exists():
            return f"Error: file '{{filename}}' not found."
        return path.read_text()


    @mcp.tool
    def write_file(filename: str, content: str) -> str:
        \"\"\"Write content to a file in the research workspace.\"\"\"
        path = Path(WORKDIR) / filename
        path.write_text(content)
        return f"File '{{filename}}' written successfully ({len(content)} chars)."


    if __name__ == "__main__":
        mcp.run(transport="stdio")
"""), encoding="utf-8")

print(f"Filesystem server written ✅ → {fs_server_path}")

---
## Step 7 — Connect via MultiServerMCPClient

In [ ]:
import sys

mcp_connections = {
    "filesystem": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(fs_server_path)],
    },
    "research_ops": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [str(server_path)],
    },
}

# Optional: replace filesystem with real npm server if available
if USE_NPM_SERVERS:
    mcp_connections["filesystem"] = {
        "transport": "stdio",
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", WORKDIR],
    }
    print("Using npm filesystem server")
else:
    print("Using Python filesystem server")

print(f"MCP connections: {list(mcp_connections.keys())}")

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

async def get_mcp_tools():
    async with MultiServerMCPClient(mcp_connections) as client:
        tools = client.get_tools()
        return tools

mcp_tools = asyncio.get_event_loop().run_until_complete(get_mcp_tools())

print(f"\nMCP tools loaded ✅ — {len(mcp_tools)} total")
for tool in mcp_tools:
    print(f"  [{tool.name}] {getattr(tool, 'description', '')[:60]}")

---
## Step 8 — Build the Gemini LangGraph Agent

In [ ]:
from langchain_core.tools import tool as lc_tool
from pathlib import Path
import json, re

# --- Fallback tools (always work, no servers needed) ---
@lc_tool
def list_research_files() -> str:
    """List all files in the research workspace."""
    files = [f.name for f in Path(WORKDIR).iterdir() if f.is_file() and not f.name.endswith('.py')]
    return "\n".join(files) if files else "(empty)"


@lc_tool
def read_research_file(filename: str) -> str:
    """Read a research note from the workspace by filename."""
    p = Path(WORKDIR) / filename
    return p.read_text() if p.exists() else f"File '{filename}' not found."


@lc_tool
def write_research_file(filename: str, content: str) -> str:
    """Write a new research note to the workspace."""
    p = Path(WORKDIR) / filename
    p.write_text(content)
    return f"Saved '{filename}' ({len(content)} chars)."


@lc_tool
def format_citation(title: str, author: str, year: int, venue: str, url: str = "") -> str:
    """Format an APA-style academic citation."""
    citation = f"{author} ({year}). {title}. {venue}."
    if url:
        citation += f" Retrieved from {url}"
    return citation


@lc_tool
def extract_keywords(text: str, n: int = 5) -> str:
    """Extract the top-N most frequent meaningful words from a text."""
    from collections import Counter
    stopwords = {"the", "a", "an", "and", "or", "in", "on", "to", "for", "of",
                 "is", "are", "was", "with", "by", "it", "this", "that", "from",
                 "as", "be", "its", "they", "their", "uses", "use"}
    words = re.findall(r'\b[a-zA-Z]{4,}\b', text.lower())
    freq = Counter(w for w in words if w not in stopwords)
    return ", ".join(word for word, _ in freq.most_common(n))


@lc_tool
def count_words(text: str) -> str:
    """Count words, sentences, and paragraphs in a text."""
    words = len(text.split())
    sentences = len([s for s in re.split(r'[.!?]+', text) if s.strip()])
    paragraphs = len([p for p in text.split('\n\n') if p.strip()])
    return f"words={words}, sentences={sentences}, paragraphs={paragraphs}"


# Use MCP tools if available, otherwise use fallback tools
if mcp_tools:
    agent_tools = mcp_tools
    print(f"Using {len(agent_tools)} MCP tools")
else:
    agent_tools = [list_research_files, read_research_file, write_research_file,
                   format_citation, extract_keywords, count_words]
    print(f"Using {len(agent_tools)} fallback tools")

In [ ]:
from langgraph.prebuilt import create_react_agent

if USE_REAL_LLM:
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(
        model="gemini-2.0-flash",
        google_api_key=GOOGLE_API_KEY,
        temperature=0
    )
    agent = create_react_agent(llm, agent_tools)
    print("Gemini ReAct agent ready ✅")
else:
    # Stub agent — rule-based orchestration for demo
    agent = None
    print("Stub mode — using manual orchestration ✅")

In [ ]:
def run_research_agent(query: str) -> dict:
    """Run the research agent (real Gemini or stub)."""
    if USE_REAL_LLM and agent:
        result = agent.invoke({"messages": [("user", query)]})
        final = result["messages"][-1].content
        return {"query": query, "answer": final, "mode": "gemini"}

    # --- Stub orchestration ---
    q = query.lower()
    tool_calls = []

    if any(k in q for k in ["list", "files", "what files"]):
        result = list_research_files.invoke({})
        tool_calls.append(("list_research_files", {}, result))
        return {"query": query, "tool_calls": tool_calls, "answer": f"Files in workspace:\n{result}", "mode": "stub"}

    if any(k in q for k in ["read", "content", "show", "open"]):
        # Guess which file
        fname = next((f for f in ["llm_overview.txt", "rag_notes.txt", "mcp_notes.txt"] if any(k in q for k in f.split("_"))), "llm_overview.txt")
        if "rag" in q: fname = "rag_notes.txt"
        if "mcp" in q: fname = "mcp_notes.txt"
        result = read_research_file.invoke({"filename": fname})
        tool_calls.append(("read_research_file", {"filename": fname}, result))
        return {"query": query, "tool_calls": tool_calls, "answer": f"Content of {fname}:\n{result}", "mode": "stub"}

    if any(k in q for k in ["citation", "cite", "reference"]):
        result = format_citation.invoke({
            "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
            "author": "Lewis, P. et al.", "year": 2020,
            "venue": "Advances in Neural Information Processing Systems (NeurIPS)",
            "url": "https://arxiv.org/abs/2005.11401"
        })
        tool_calls.append(("format_citation", {}, result))
        return {"query": query, "tool_calls": tool_calls, "answer": f"Formatted citation:\n{result}", "mode": "stub"}

    if any(k in q for k in ["keyword", "topic", "extract", "analyze"]):
        # Read a file and extract keywords
        content = read_research_file.invoke({"filename": "rag_notes.txt"})
        keywords = extract_keywords.invoke({"text": content, "n": 7})
        tool_calls.append(("read_research_file", {"filename": "rag_notes.txt"}, content))
        tool_calls.append(("extract_keywords", {"text": "...", "n": 7}, keywords))
        return {"query": query, "tool_calls": tool_calls, "answer": f"Top keywords from rag_notes.txt:\n{keywords}", "mode": "stub"}

    if any(k in q for k in ["write", "save", "create", "note"]):
        fname = "agent_summary.txt"
        content = f"Agent summary generated for query: '{query}'\n\nKey findings: The research assistant demonstrates MCP + LangGraph + Gemini integration."
        result = write_research_file.invoke({"filename": fname, "content": content})
        tool_calls.append(("write_research_file", {"filename": fname}, result))
        return {"query": query, "tool_calls": tool_calls, "answer": result, "mode": "stub"}

    # Fallback
    return {"query": query, "tool_calls": [], "answer": "I'm not sure which tool to use. Try: 'list files', 'read mcp notes', 'format a citation', or 'extract keywords'.", "mode": "stub"}


print("Research agent runner defined ✅")

---
## Step 9 — Demo Queries

In [ ]:
DEMO_QUERIES = [
    "List all files in the research workspace.",
    "Read the content of mcp_notes.txt.",
    "Format a citation for the RAG paper by Lewis et al. 2020 from NeurIPS.",
    "Extract the top keywords from the RAG notes file.",
    "Write a summary note about the agent architecture.",
]

sep = "=" * 65

for query in DEMO_QUERIES:
    result = run_research_agent(query)
    print(sep)
    print(f"QUERY : {result['query']}")
    if 'tool_calls' in result:
        for tc in result['tool_calls']:
            print(f"TOOL  : {tc[0]}({list(tc[1].keys()) if tc[1] else ''})")
    print(f"ANSWER: {result['answer']}")
    print()

---
## Step 10 — Verify the Full Agent Flow

In [ ]:
# Verify the new file was written
print("Files after agent run:")
for f in sorted(Path(WORKDIR).iterdir()):
    if not f.name.endswith(".py"):
        print(f"  {f.name} ({f.stat().st_size} bytes)")

print()
# Read back the summary written by the agent
summary_path = Path(WORKDIR) / "agent_summary.txt"
if summary_path.exists():
    print("Content of agent_summary.txt:")
    print(summary_path.read_text())

In [ ]:
# Test custom server tools independently
print("=== Custom MCP Server Tool Tests ===")

# Test format_citation
citation = format_citation.invoke({
    "title": "Attention Is All You Need",
    "author": "Vaswani, A. et al.",
    "year": 2017,
    "venue": "NeurIPS",
    "url": "https://arxiv.org/abs/1706.03762"
})
print(f"\nformat_citation:\n  {citation}")

# Test count_words
sample = (Path(WORKDIR) / "rag_notes.txt").read_text()
word_count = count_words.invoke({"text": sample})
print(f"\ncount_words (rag_notes.txt):\n  {word_count}")

# Test extract_keywords
keywords = extract_keywords.invoke({"text": sample, "n": 6})
print(f"\nextract_keywords (rag_notes.txt):\n  {keywords}")

---
## Summary

### What was built

| Component | Description |
|-----------|-------------|
| **Theme** | Research Assistant — manages notes, citations, keyword extraction |
| **MCP Server 1** | `filesystem_server.py` (FastMCP) — `list_files`, `read_file`, `write_file` |
| **MCP Server 2** | `research_ops_server.py` (FastMCP) — `format_citation`, `count_words`, `extract_keywords`, `ping` |
| **Third-party** | `@modelcontextprotocol/server-filesystem` (npm, optional) |
| **Client** | `MultiServerMCPClient` — connects both servers, aggregates tools |
| **Agent** | LangGraph `create_react_agent` with Gemini (or stub orchestration) |
| **Queries** | List files, read notes, format citation, extract keywords, write summary |

### Agent Architecture

```
User Query
    └──► ReAct Agent (Gemini gemini-2.0-flash)
              │
              │ observe → plan → act loop
              │
              ├──► [filesystem] list_files()
              ├──► [filesystem] read_file(filename)
              ├──► [filesystem] write_file(filename, content)
              ├──► [research_ops] format_citation(...)
              ├──► [research_ops] extract_keywords(text, n)
              └──► [research_ops] count_words(text)
                        ↓
                  Final answer (LLM-generated)
```

### Key Learnings

1. **FastMCP** makes custom server creation trivial — just `@mcp.tool` decorators
2. **MultiServerMCPClient** aggregates tools from multiple servers into a single list
3. **create_react_agent** gives the LLM full autonomy to decide tool order and count
4. **STDIO transport** — each server runs as a subprocess, dies when client disconnects
5. **Composability** — adding a new server = one dict entry in `mcp_connections`